# Joint histograms: observed and PARSEC droplet number

Two views of the hourly comparison, per site:

1. **Parcel vs observed Nd**: joint histogram of log Nd (observed on x, parcel on y) with the 1:1 and factor-2 lines, the marginal distributions, and the median ratio, r and log-log slope.
2. **Nd vs aerosol number N(>D)**, side by side for the observations and for the parcel model on the same hours (the susceptibility view): joint histogram, bin medians and the binned-median regression slope S = d ln Nd / d ln N(>D), as in the observational susceptibility recipe (log10, equal-number bins, regression through the bin medians).

Data: `results/nd_comparison_hours.csv` (from `compare_nd.py`; observed-updraft variant). Optional extra columns from `results/cloudnet_quality/<site>_flags.csv` (drizzle, LWP, depth, visibility, rain) for screening with a pandas query. N(>D) at a cutoff other than 100 nm is read from the observational built files (`$OBS_REPO`).

Colours are counts (sequential colormap); the only diverging colormap is for the difference panel in section 4.

## 1 · Settings

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

warnings.filterwarnings("ignore", category=RuntimeWarning)
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
OBS_REPO = os.environ.get("OBS_REPO", "/nobackup/proj/disk/bolinc/personal/petchu/code/obs-ccn-nd-susceptibility")

WEIGHTING = "flux"          # "flux" or "pdf"
SITES = None                # None = all sites in the comparison file, or e.g. ["puijo", "pallas", "lasic"]
QUERY = None                # pandas query on hours + flags, e.g. "visibility_m < 200 and rain_mmh == 0"
                            # or "depth < 800 and abs(cbh_minus_lcl) <= 300" or "lwp_gm2 >= 30"
CUTOFF_NM = 100             # D for N(>D) in section 3 (100 uses the N100 column; others need OBS_REPO)
NBINS = 40                  # histogram bins per axis (log space)
N_EQUAL_BINS = 20           # equal-number bins for the binned-median slope
SAVE = False                # True: save figures to results/joint_histograms/
OUTDIR = os.path.join(REPO, "results", "joint_histograms")
print("repo:", REPO)

## 2 · Load the hourly comparison

In [ ]:
t = pd.read_csv(os.path.join(REPO, "results", "nd_comparison_hours.csv"), parse_dates=["hour"])
t = t[(t["variant"] == "obs") & (t["weighting"] == WEIGHTING) & t["usable"]]
t = t[(t["Nd_obs"] > 0) & (t["Nd_parcel"] > 0)]
if SITES:
    t = t[t["site"].isin(SITES)]

def with_flags(d, site):
    p = os.path.join(REPO, "results", "cloudnet_quality", f"{site}_flags.csv")
    if not os.path.exists(p):
        return d
    f = pd.read_csv(p, parse_dates=["hour"]).set_index("hour")
    d = d.set_index("hour").join(f, how="left", rsuffix="_flag").reset_index()
    if "liquid_depth_cat_m" in d or "depth_built_m" in d:
        d["depth"] = d.get("liquid_depth_cat_m", pd.Series(np.nan, index=d.index)).fillna(
            d.get("depth_built_m", pd.Series(np.nan, index=d.index)))
    return d

parts = []
for s, g in t.groupby("site"):
    g = with_flags(g, s)
    if QUERY:
        try:
            g = g.query(QUERY)
        except Exception as ex:      # a site without the queried columns is skipped
            print(f"{s}: query not applicable ({str(ex)[:60]})"); continue
    parts.append(g)
D = pd.concat(parts, ignore_index=True)
sites = sorted(D["site"].unique())
print(f"{len(D)} hours at {len(sites)} sites" + (f" after query '{QUERY}'" if QUERY else ""))
D.groupby("site").size().rename("hours").to_frame().T

## 3 · Helpers

In [ ]:
def fit(x, y):
    lx, ly = np.log10(x), np.log10(y)
    q = ly - lx
    return dict(n=len(x), ratio=10**np.median(q), q25=10**np.percentile(q, 25), q75=10**np.percentile(q, 75),
                r=np.corrcoef(lx, ly)[0, 1] if len(x) > 2 else np.nan,
                slope=np.polyfit(lx, ly, 1)[0] if len(x) > 2 else np.nan)

def binned_median_slope(x, y, nbins=N_EQUAL_BINS):
    # observational recipe: log10, equal-number bins (sort + array_split), median per bin, linear fit
    lx, ly = np.log10(x), np.log10(y)
    o = np.argsort(lx)
    bx = np.array([np.median(c) for c in np.array_split(lx[o], nbins) if len(c)])
    by = np.array([np.median(c) for c in np.array_split(ly[o], nbins) if len(c)])
    s, i = np.polyfit(bx, by, 1) if len(bx) > 2 else (np.nan, np.nan)
    return 10**bx, 10**by, s, i

def edges(*arrays):
    v = np.log10(np.concatenate([np.asarray(a, float) for a in arrays]))
    return np.logspace(np.floor(v.min() * 2) / 2, np.ceil(v.max() * 2) / 2, NBINS + 1)

def hist2d(ax, x, y, bx, by, vmax=None):
    h, _, _ = np.histogram2d(x, y, bins=[bx, by])
    m = ax.pcolormesh(bx, by, np.where(h > 0, h, np.nan).T, cmap="viridis", norm=LogNorm(vmin=1, vmax=vmax or max(1, h.max())))
    ax.set(xscale="log", yscale="log")
    return m, h

def save(fig, name):
    if SAVE:
        os.makedirs(OUTDIR, exist_ok=True)
        fig.savefig(os.path.join(OUTDIR, name), dpi=150, bbox_inches="tight")

## 4 · Parcel vs observed Nd: all sites

In [ ]:
nc = 5; nr = int(np.ceil(len(sites) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(3.4 * nc, 3.4 * nr), squeeze=False)
for ax, s in zip(axs.flat, sites):
    g = D[D["site"] == s]
    b = edges(g["Nd_obs"], g["Nd_parcel"])
    m, _ = hist2d(ax, g["Nd_obs"], g["Nd_parcel"], b, b)
    lim = [b[0], b[-1]]
    ax.plot(lim, lim, "k-", lw=0.8)
    for f in (0.5, 2):
        ax.plot(lim, [l * f for l in lim], "k:", lw=0.6)
    st = fit(g["Nd_obs"].to_numpy(), g["Nd_parcel"].to_numpy())
    ax.text(0.03, 0.97, f"n {st['n']}\nratio {st['ratio']:.2f}\nr {st['r']:.2f}, slope {st['slope']:.2f}",
            transform=ax.transAxes, va="top", fontsize=7, bbox=dict(fc="white", ec="0.8", alpha=0.85))
    ax.set(xlim=lim, ylim=lim, title=s); ax.set_aspect("equal")
for ax in axs.flat[len(sites):]:
    ax.set_visible(False)
for ax in axs[:, 0]: ax.set_ylabel("parcel Nd (cm$^{-3}$)")
for ax in axs[-1]: ax.set_xlabel("observed Nd (cm$^{-3}$)")
fig.suptitle(f"Parcel vs observed Nd, joint histograms ({WEIGHTING} weighting; colour = hours)")
fig.tight_layout(); save(fig, f"parcel_vs_obs_grid_{WEIGHTING}.png")

## 5 · One site in detail: joint histogram with marginals

In [ ]:
SITE = sites[0]          # change to the site to look at
g = D[D["site"] == SITE]
b = edges(g["Nd_obs"], g["Nd_parcel"])
fig = plt.figure(figsize=(6.5, 6.5))
gs = fig.add_gridspec(2, 2, width_ratios=(4, 1), height_ratios=(1, 4), hspace=0.05, wspace=0.05)
ax = fig.add_subplot(gs[1, 0]); axx = fig.add_subplot(gs[0, 0], sharex=ax); axy = fig.add_subplot(gs[1, 1], sharey=ax)
m, _ = hist2d(ax, g["Nd_obs"], g["Nd_parcel"], b, b)
lim = [b[0], b[-1]]
ax.plot(lim, lim, "k-", lw=0.8)
for f in (0.5, 2):
    ax.plot(lim, [l * f for l in lim], "k:", lw=0.6)
axx.hist(g["Nd_obs"], bins=b, color="0.3"); axy.hist(g["Nd_parcel"], bins=b, orientation="horizontal", color="#1f77b4")
axx.axis("off"); axy.axis("off")
st = fit(g["Nd_obs"].to_numpy(), g["Nd_parcel"].to_numpy())
ax.set(xlim=lim, ylim=lim, xlabel="observed Nd (cm$^{-3}$)", ylabel="parcel Nd (cm$^{-3}$)")
ax.text(0.03, 0.97, f"{SITE}: n {st['n']}, ratio {st['ratio']:.2f} ({st['q25']:.2f}-{st['q75']:.2f})\n"
        f"r {st['r']:.2f}, slope {st['slope']:.2f}", transform=ax.transAxes, va="top", fontsize=8,
        bbox=dict(fc="white", ec="0.8", alpha=0.85))
fig.colorbar(m, ax=axy, label="hours", fraction=0.6)
save(fig, f"parcel_vs_obs_{SITE}_{WEIGHTING}.png")

## 6 · Nd vs N(>D): observations and parcel side by side

In [ ]:
def n_above(site, hours, cutoff):
    # N(>D) per hour: the N100 column at 100 nm, else interpolated from the built file's CCN(time, radius)
    if cutoff == 100:
        return None
    import xarray as xr, sys
    sys.path.insert(0, REPO)
    from sites import SITES as SITE_TABLE
    p = os.path.join(OBS_REPO, SITE_TABLE[site]["built"])
    with xr.open_dataset(p) as ds:
        tt = pd.DatetimeIndex(ds["time"].values).floor("h")
        dim = [d for d in ds["CCN"].dims if d != "time"][0]
        dd = ds[dim].values.astype(float) * (2 if dim == "radius" else 1)          # diameter, nm
        c = pd.DataFrame(ds["CCN"].values, index=tt)
    c = c[~c.index.duplicated()].reindex(hours)
    k = np.argmin(np.abs(dd - cutoff))
    return c.iloc[:, k].to_numpy(float)

def two_panel(site):
    g = D[D["site"] == site].copy()
    na = n_above(site, pd.DatetimeIndex(g["hour"]), CUTOFF_NM)
    g["N"] = g["N100"] if na is None else na
    g = g[(g["N"] > 0)]
    bx = edges(g["N"]); by = edges(g["Nd_obs"], g["Nd_parcel"])
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.4), sharex=True, sharey=True)
    vmax = max(np.histogram2d(g["N"], g[k], bins=[bx, by])[0].max() for k in ("Nd_obs", "Nd_parcel"))
    for ax, k, lab, col in zip(axs, ("Nd_obs", "Nd_parcel"), ("observed", "parcel"), ("k", "#d62728")):
        m, _ = hist2d(ax, g["N"], g[k], bx, by, vmax=vmax)
        mx, my, s, i = binned_median_slope(g["N"].to_numpy(), g[k].to_numpy())
        ax.plot(mx, my, "o", color=col, ms=4, mec="white", mew=0.5, label="bin medians")
        xx = np.array([bx[0], bx[-1]])
        ax.plot(xx, 10**(i + s * np.log10(xx)), "-", color=col, lw=1.2, label=f"S = {s:.2f}")
        ax.plot(xx, xx, color="0.5", lw=0.6, ls="--", label="1:1")
        ax.set(title=f"{site}: {lab} (n {len(g)})", xlabel=f"N(>{CUTOFF_NM} nm) (cm$^{{-3}}$)")
        ax.legend(fontsize=7, loc="upper left")
    axs[0].set_ylabel("Nd (cm$^{-3}$)")
    fig.colorbar(m, ax=axs, label="hours", fraction=0.03)
    save(fig, f"nd_vs_n{CUTOFF_NM}_{site}_{WEIGHTING}.png")
    return fig

for s in sites:
    two_panel(s)

## 7 · Difference of the two joint histograms

Where the parcel model puts hours that the observations do not (red) and vice versa (blue), on the same N(>D)–Nd grid, as the difference of normalised histograms. Diverging colormap, since the difference is signed.

In [ ]:
SITE = sites[0]
g = D[D["site"] == SITE].copy()
na = n_above(SITE, pd.DatetimeIndex(g["hour"]), CUTOFF_NM)
g["N"] = g["N100"] if na is None else na
g = g[g["N"] > 0]
bx = edges(g["N"]); by = edges(g["Nd_obs"], g["Nd_parcel"])
ho = np.histogram2d(g["N"], g["Nd_obs"], bins=[bx, by])[0]; hp = np.histogram2d(g["N"], g["Nd_parcel"], bins=[bx, by])[0]
diff = (hp / hp.sum() - ho / ho.sum()) * 100
v = np.nanmax(np.abs(diff))
fig, ax = plt.subplots(figsize=(5.6, 4.6))
m = ax.pcolormesh(bx, by, np.where((ho + hp) > 0, diff, np.nan).T, cmap="RdBu_r", vmin=-v, vmax=v)
ax.set(xscale="log", yscale="log", xlabel=f"N(>{CUTOFF_NM} nm) (cm$^{{-3}}$)", ylabel="Nd (cm$^{-3}$)",
       title=f"{SITE}: parcel minus observed (% of hours)")
fig.colorbar(m, ax=ax, label="percentage points")
save(fig, f"nd_vs_n{CUTOFF_NM}_diff_{SITE}_{WEIGHTING}.png")